In [ ]:
# --- Task10 drift-path notebook: 2cm drift @ 0.1mm, enforcement-free ------------
# Trimmed analysis notebook: DRIFT PATHS ONLY (no field/potential diagnostics).
# Store: store_task10_drift_2cm_01mm  (test/run-task10-drift-2cm-01mm.sh output).
# 100 electrons launched over the 0.6mm inter-pixel gap band (10x10 grid) at
# z=29.9mm; they drift down and halt at z~9.85-9.95mm just above the pad plane.
# Pure numpy/matplotlib (mplhep 'CMS'), no pochoir import -> runs anywhere the
# store is present.
path_to_data = 'store_task10_drift_2cm_01mm'
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patches
try:                                    # CMS style if available; else plain mpl
    import mplhep as hep; hep.style.use('CMS')
except ImportError:
    pass

def _load(key):
    """Flat npz loader: return the single array stored under `key`.npz."""
    f = np.load('/'.join([path_to_data, key + '.npz'])); return f[f.files[0]]

_starts = _load('starts/drift3d')                 # (100,3) launch points x,y,z (mm)
NGRID   = int(round(len(_starts)**0.5))           # 10
_p      = _load('paths/drift3d')                  # (100,800,3) electron x tick x xyz
NSAMP   = _p.shape[1]                             # 800
paths   = _p.reshape(NGRID, NGRID, NSAMP, 3)      # (x-idx, y-idx, tick, xyz)
LAUNCH  = np.round(np.unique(_starts[:, 0]), 3)   # 10 launch x/y values (mm)

# Pad-plane conductor mask (pad metal at z-node 100 = z=10.0mm); 0.1mm spacing.
SPACING = 0.1
pad2 = _load('boundary/drift_2cm')[:, :, 100] > 0  # (44,44) True where pad metal

print(f'paths {paths.shape} | launch {NGRID}x{NGRID}, x,y in '
      f'[{LAUNCH.min()},{LAUNCH.max()}] mm, z={np.unique(_starts[:,2])[0]} mm')
print(f'pad mask {pad2.shape}, {int(pad2.sum())} metal cells at z-node 100')


In [ ]:
# --- x-z drift-path slice at a fixed launch-y (through the gap centre) ----------
J = NGRID // 2                          # launch-y index near the gap centre
y_slice = LAUNCH[J]
print(f'x-z slice at launch y = {y_slice} mm  ({NGRID} paths)')

fig, ax = plt.subplots()
for i in range(NGRID):
    ax.plot(paths[i, J, :, 0], paths[i, J, :, 2], lw=1)

# Data-driven pad x-edges from the boundary mask (row through the pad, y-node 10):
# each 0.1mm cell edge is (node +/- 0.5)*spacing.  Read the metal segments so the
# rectangles track the actual conductor rather than hard-coded numbers.
_row = np.where(pad2[:, 10])[0]
_segs = []
if len(_row):
    _s = _p0 = _row[0]
    for _v in _row[1:]:
        if _v != _p0 + 1:
            _segs.append((_s, _p0)); _s = _v
        _p0 = _v
    _segs.append((_s, _p0))
pixel_pads = [((a - 0.5) * SPACING, (b + 0.5) * SPACING) for a, b in _segs]
print(f'pad x segments (nodes) {_segs} -> mm edges {[(round(l,2),round(r,2)) for l,r in pixel_pads]}')

z_bottom, z_top = 9.9, 10.0
for x_left, x_right in pixel_pads:
    ax.add_patch(patches.Rectangle((x_left, z_bottom), x_right - x_left,
                 z_top - z_bottom, linewidth=2, edgecolor='red',
                 facecolor='white', alpha=0.4))
ax.set_ylim(9., 12)
ax.set_xlabel('x (mm)')
ax.set_ylabel('z (mm)')
ax.set_title(f'task10 drift paths, launch y = {y_slice} mm')
plt.grid()


In [ ]:
# --- x-y top-down trajectory view over the pad footprint -----------------------
# All 100 paths projected onto x-y: shows the transverse focusing toward the pad
# centre (~2.2, 2.2 mm) as the electrons descend into the gap.
fig, ax = plt.subplots()
_extent = [0, pad2.shape[0] * SPACING, 0, pad2.shape[1] * SPACING]
ax.imshow(pad2.T, origin='lower', extent=_extent, cmap='Greys', alpha=0.35)
for i in range(NGRID):
    for j in range(NGRID):
        ax.plot(paths[i, j, :, 0], paths[i, j, :, 1], lw=0.8)
ax.set_xlabel('x (mm)')
ax.set_ylabel('y (mm)')
ax.set_title('task10 drift paths (top-down x-y) over pad footprint')
ax.set_aspect('equal')


In [ ]:
# --- top-down landing map: pad (blue) vs gap (red) -----------------------------
# endtags are all-zero for this velocity-engine run, so pad-vs-gap landing is
# derived from the boundary mask + the last-moving endpoint (paths are zero-padded
# after a charge stops, so [...,-1] is unreliable; use the np.diff>1e-9 trick).
fig, ax = plt.subplots()
ax.imshow(pad2.T, origin='lower', extent=[0, pad2.shape[0]*SPACING, 0,
          pad2.shape[1]*SPACING], cmap='Greys', alpha=0.35)

n_gap = 0
for i in range(NGRID):
    for j in range(NGRID):
        tr = paths[i, j]
        mv = np.where(np.abs(np.diff(tr[:, 0])) + np.abs(np.diff(tr[:, 1]))
                      + np.abs(np.diff(tr[:, 2])) > 1e-9)[0]
        xe, ye, ze = tr[mv[-1] + 1] if len(mv) else tr[-1]
        onpad = pad2[int(round(xe / SPACING)) % pad2.shape[0],
                     int(round(ye / SPACING)) % pad2.shape[1]]
        if not onpad:
            n_gap += 1
        ax.plot(xe, ye, 'o', ms=5, color='tab:blue' if onpad else 'tab:red')

ax.set_xlabel('x (mm)')
ax.set_ylabel('y (mm)')
ax.set_title(f'task10 landings: {NGRID*NGRID - n_gap} on pad (blue), '
             f'{n_gap} in gap (red)')
ax.set_aspect('equal')
print(f'gap landings: {n_gap} of {NGRID*NGRID}')


In [ ]:
# --- 3D overview of all 100 drift trajectories --------------------------------
# Live analogue of the runner's drift_paths_3d.png (kept in sync with the array).
fig = plt.figure(figsize=(9, 9))
ax = fig.add_subplot(111, projection='3d')
for i in range(_p.shape[0]):
    ax.plot(_p[i, :, 0], _p[i, :, 1], _p[i, :, 2], lw=0.7)
ax.set_xlabel('x (mm)')
ax.set_ylabel('y (mm)')
ax.set_zlabel('z (mm)')
ax.set_title('task10 drift paths (3D)')


**Note on landing classification.** This is a velocity-engine drift run, so
`paths/drift3d_endtag` is all-zero (no pad/FR4 collection tags). Pad-vs-gap
landing above is therefore derived from the conductor boundary mask
(`boundary/drift_2cm[:, :, 100]`) evaluated at each path's *last moving* endpoint,
not from endtags. Charges launch at z=29.9 mm and halt at z ~ 9.85-9.95 mm just
above the pad plane (z=10.0 mm).